# Geico Quote Decision Predictor

---

## 1 · Problem Overview
The objective of this project is to predict whether or not a customer will approve or deny an insurance quote using machine learning techniques. I will execute an end-to-end analytical approach and apply machine learning models to internal company sales data to help the business gain a better understanding of the patterns and data dimensions demystifying customer needs and respond more strategically through informed and evaluated business recommendations.

---
## 2 · Data Collection & Cleaning

Load the proprietary quote dataset, then clean:
- Rename columns for clarity (snake_case)
- Check for missing values
- Apply data transformations:
  - Unit normalization to mitigate outliers
  - Log transformation to reduce skewness in price and features  
- Detect and address outliers (IQR)

#### 2.1 · Imports

In [1]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# -----------------------------
# Add project root to sys.path for module imports
# -----------------------------
root = "/Users/phillipsmith/Desktop/Python/quote_decision_predictor"
sys.path.append(root)

# -----------------------------
# Import external python modules
# -----------------------------
from pathlib import Path

from src.preprocessing import data_preparation
# from src.models import regression_models
# from src.viz import eda

# from sklearn.model_selection import train_test_split
# from sklearn.feature_selection import SelectKBest, f_regression, RFE
# from sklearn.linear_model import LinearRegression
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import make_pipeline
# from sklearn.model_selection import KFold, cross_val_predict
# from sklearn.neighbors import KNeighborsRegressor
# from sklearn.metrics import mean_squared_error, r2_score

# from math import sqrt

#### 2.2 · Configuration & Data Loading

In [2]:
# -----------------------------
# Display full dataframes
# -----------------------------
pd.set_option('display.max_rows', None) # reset_option to compact dataframe view
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.6f}'.format)

# -----------------------------
# Define path to raw Excel data source
# -----------------------------
excel_path = Path(
    '/Users/phillipsmith/Desktop/Python/quote_decision_predictor/src/data/raw/P1_Geico_Quote_Data.xlsx'
)

# -----------------------------
# Read raw data from Excel source
# -----------------------------
df_raw = pd.read_excel(excel_path, skiprows=0)
print(f"Raw DataFrame shape: {df_raw.shape}")

df_raw.columns

Raw DataFrame shape: (5000, 17)


Index(['QuoteId', 'ZipCode', 'QuoteType', 'QuoteSts', 'PHolderAge',
       'PHolderGender', 'PHolderMSts', 'PHolderHomeSts', 'PHolderDriveExp',
       'PHolderEdu', 'PHolderDriveRisk', 'CarAge', 'CarType', 'OwnStatus',
       'DailyAvgMiles', 'AnnualMileageRange', 'NoOfVehicles'],
      dtype='object')

#### 2.3 · Data Cleaning

Column headers are not standardized, so I need to rename them for clarity.

In [9]:
df_cleaned = data_preparation.clean_data(df_raw)

# # statistical summary
df_cleaned.describe()

,ID,ZIP,TYPE_Q,STAT_Q,AGE_PH,SEX,HOME_STAT,DRIVE_XP,EDU_PH,DRIVE_RISK,AGE_CAR,OWN_CAR,AVG_MILE_DAILY,ANNUAL_MILE,NUM_CAR,MARR_STAT_Divorced,MARR_STAT_Married,MARR_STAT_Single,TYPE_CAR_Luxury,TYPE_CAR_Minivan,TYPE_CAR_SUV,TYPE_CAR_Sedan,TYPE_CAR_Sports
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,2500.500000,102083.950000,0.586000,0.685800,26.464000,0.451600,0.501600,0.303400,2.019000,1.998400,2.008600,0.481400,2.019800,2.019800,1.499600,0.104400,0.500600,0.395000,0.098400,0.266800,0.260600,0.273000,0.101200
std,1443.520003,4874640.499422,0.492598,0.464243,11.599621,0.497702,0.500047,0.812084,1.212325,1.237539,1.166361,0.499704,1.146678,1.146678,1.116893,0.305809,0.500050,0.488900,0.297885,0.442331,0.439006,0.445545,0.301624
min,1.000000,6375.000000,0.000000,0.000000,18.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1250.750000,33033.000000,0.000000,0.000000,21.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2500.500000,33147.000000,1.000000,1.000000,23.000000,0.000000,1.000000,0.000000,2.000000,2.000000,2.000000,0.000000,2.000000,2.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,3750.250000,33175.000000,1.000000,1.000000,26.000000,1.000000,1.000000,0.000000,3.000000,3.000000,3.000000,1.000000,3.000000,3.000000,3.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,0.000000
max,5000.000000,344722250.000000,1.000000,1.000000,103.000000,1.000000,1.000000,3.000000,4.000000,4.000000,4.000000,1.000000,4.000000,4.000000,3.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


#### 2.4 · Missing Values
Check for nulls. Apply imputation as needed (mean/median for numeric, mode for categorical).

In [4]:
print(df_cleaned[df_cleaned.columns].isna().sum()) # None

ID                    0
ZIP                   0
TYPE_Q                0
STAT_Q                0
AGE_PH                0
SEX                   0
HOME_STAT             0
DRIVE_XP              0
EDU_PH                0
DRIVE_RISK            0
AGE_CAR               0
OWN_CAR               0
AVG_MILE_DAILY        0
ANNUAL_MILE           0
NUM_CAR               0
MARR_STAT_Divorced    0
MARR_STAT_Married     0
MARR_STAT_Single      0
TYPE_CAR_Luxury       0
TYPE_CAR_Minivan      0
TYPE_CAR_SUV          0
TYPE_CAR_Sedan        0
TYPE_CAR_Sports       0
dtype: int64


No missing values found in the dataframe.

#### 2.5 · Data Transformations

Apply unit normalization, log transforms, outlier detection, and one hot encoding to numeric features to account for outliers, reduce skewness, and improve feature quality of independent and dependent variables.

In [5]:
df_transformed = data_transformation.transform_data(df_cleaned.copy())
df_transformed.head()

NameError: name 'data_transformation' is not defined

---
## 3 · Exploratory Data Analysis (EDA)

#### 3.1 · Raw Variable Distributions

In [ ]:
"""RAW DISTRIBUTION PLOTS"""
var_raw = ''
var_log = var_raw + "_log"

# normal histplot
fig, ax = plt.subplots(figsize=(10,4))
plt.title(var_raw)
sns.histplot(df_transformed[var_raw], bins='auto', ax=ax)

# log transformed histplot
df_transformed[var_log] = np.log1p(df_transformed[var_raw])
fig, ax = plt.subplots(figsize=(10,4))
plt.title(var_log)
sns.histplot(df_transformed[var_log], bins='auto', ax=ax)

#### 3.2 · Generate EDA Plots
- Generate histograms, count plots, and a correlation matrix for numeric features (more info on bin parameters: 
(https://numpy.org/doc/stable/reference/generated/numpy.histogram_bin_edges.html#numpy.histogram_bin_edges)
- Look for skewness → candidates for log and Box-Cox transforms
    - bins='fd' (Freedman-Diaconis) is often good for skewed data
- Plots are exported to (`docs`) directory for review.

In [ ]:
"""HISTOGRAMS, COUNT, SCATTER PLOTS, & CORRELATION MATRIX"""

df_correlated = eda.explore_data(df_transformed)
df_correlated.head()

While correlation is a useful first-pass filter, I will also consider feature selection methods to capture non-linear relationships between features that correlation may miss.

---
## 4 · Feature Engineering

- Determine importance of features by applying feature selection methods
- Use error metrics (RMSE) to evaluate feature subsets and select the optimal set for modeling

#### 4.0 · Baseline Error Check

In [ ]:
engineer = feature_engineering.FeatureEngineer(df_correlated, n_splits=10, n_neighbors=10)
X = df_correlated.drop(columns='')
y = df_correlated['']
rmse, r2 = engineer.evaluate_error(X, y)
print(f"      BASELINE ERROR: RMSE={rmse} --> R^2={r2}")

#### 4.1 · Feature Selection Using a Wrapper

Sequential Forward Selection (SFS) is a wrapper method that iteratively adds features to the model based on performance improvement. It evaluates the model's performance (using RMSE) after adding each feature and selects the one that provides the best improvement. This process continues until the specified number of features (k_features=9) is reached.

In [ ]:
df_engineered = feature_engineering.engineer_features(df_correlated)
df_engineered.head()

#### 4.2 · Univariate: SelectKBest with f_regression

In [ ]:
X = df_correlated.drop(columns=[''])
y = df_correlated['']
n_features = 9

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=3)

"""METHOD 1: UNIVARIATE SELECTION"""
# Apply SelectKBest with chi2
select_k_best = SelectKBest(score_func=f_regression, k=n_features)
X_train_k_best = select_k_best.fit_transform(X_train, y_train)
# print("SELECTED FEATURE (METHOD 1): \n", X_train.columns[select_k_best.get_support()])

In [ ]:
# select features from SFS output
X = df_correlated.drop(columns=['price'])[[

]]

y = df_correlated['']

# RMSE and R^2
engineer.evaluate_error(X, y)

#### 4.3 · Recursive Feature Elimination (RFE) with Linear Regression

In [ ]:
"""METHOD 2: RECURSIVE FEATURE ELIMINATION"""
X = df_correlated.drop(columns=[''])
y = df_correlated['']
n_features = 5

# Apply RFE with logistic regression
linear_regression = LinearRegression()
rfe = RFE(linear_regression, n_features_to_select=n_features)
X_train_rfe = rfe.fit_transform(X_train, y_train)
print("\nSELECTED FEATURES (METHOD 2): \n", X_train.columns[rfe.get_support()])

In [ ]:
# select features from SFS output
X = df_correlated.drop(columns=[''])[[
    
]]

y = df_correlated['']

# RMSE and R^2
engineer.evaluate_error(X, y)

#### 4.4 · Tree-Based Feature Importance

In [ ]:
"""METHOD 3: TREE-BASED FEATURE IMPORTANCE"""
X = df_correlated.drop(columns=[''])
y = df_correlated['']
n_features = 9

# Train random forest and get feature importances
random_forest_regressor = RandomForestRegressor()
random_forest_regressor.fit(X_train, y_train)
importances = random_forest_regressor.feature_importances_

# Display feature importances
feature_importances = pd.Series(importances, index=X_train.columns)
# print("\nSELECTED FEATURES (METHOD 3): \n", feature_importances.sort_values(ascending=False))

In [ ]:
# select features from SFS output
X = df_correlated.drop(columns=[''])[[
    
]]

y = df_correlated['']

# RMSE and R^2
engineer.evaluate_error(X, y)

#### 4.5 DataFrame Modeling
Selected independent variables (X) and a dependent variable (y). 

Display summary statistics for the final feature set.

In [ ]:
df_modeled = df_engineered

df_modeled.describe()

---
## 5 · Data Splitting

#### 5.1 · Train/Test Split
Split data into training and testing sets. Start with 80/20, then experiment with 50/50 and 95/5 splits.

Example of 20/80 train-test-split --> random_forest = regression_models.Regression(df_modeled, `test_size=.2`, random_state=3)

#### 5.2 · Strategy 
- **Multiple split ratios** — 80/20 (default), 50/50 (stress test), 95/5 (data-hungry models)
- **Stratified sampling** — bin `price` into quantiles and stratify to ensure representative splits 

---
## 6 · Baseline Model

#### 6.1 · Linear Regression
Train OLS linear regression. Evaluate with RMSE, MAE, R².

In [ ]:
# LINEAR REGRESSION (20/80 train-test-split)

linear_regression = regression_models.Regression(df_modeled, test_size=.2, random_state=3)
X = df_modeled.drop(columns=[])
y = df_modeled['']
y_pred, y_test, X_test, model, X = linear_regression.linear_regression(X, y)
linear_regression.print_results(y_pred, y_test, X_test, model)

---
## 7 · Hyperparameter Tuning

#### 7.1 · Model 2 — Random Forest
Train a Random Forest regressor. Compare performance against linear regression.

In [ ]:
# RANDOM FOREST REGRESSION (20/80 train-test-split)

random_forest = regression_models.Regression(df_modeled, test_size=.2, random_state=3)
X = df_modeled.drop(columns=[])
y = df_modeled['']
y_pred, y_test, X_test, model, X = random_forest.random_forest(X, y)
random_forest.print_results(y_pred, y_test, X_test, model)

#### 7.2 · Model 3 — Ridge & Lasso Regression
Train Ridge and Lasso variants. Evaluate how regularization affects prediction accuracy.

In [ ]:
# RIDGE REGRESSION

ridge_regression = regression_models.Regression(df_modeled, test_size=.2, random_state=3)
X = df_modeled.drop(columns=[])
y = df_modeled['']
y_pred, y_test, X_test, model, X = ridge_regression.ridge_regression(X, y)
ridge_regression.print_results(y_pred, y_test, X_test, model)

Lasso regression was also executed in my ML pipeline, but it did not perform as well as Ridge regression.

#### 7.3 · Scaling & Normalization
Apply `StandardScaler` and/or `MinMaxScaler` to features. Re-train models and compare results.

In [ ]:
# RANDOM FOREST SCALED

scaler = StandardScaler()
scaled_array = scaler.fit_transform(df_modeled)
df_scaled = pd.DataFrame(scaled_array, columns=df_modeled.columns)

random_forest = regression_models.Regression(df_scaled, test_size=.2, random_state=3)
X = df_scaled.drop(columns=[])
y = df_scaled['']
y_pred, y_test, X_test, model, X = random_forest.random_forest(X, y)
random_forest.print_results(y_pred, y_test, X_test, model)

Scaling the features using StandardScaler slightly decreased performance of the Random Forest model suggesting that the model may not be sensitive to feature scaling, which is common for tree-based models. However, I will continue to explore scaling and normalization techniques with other models to see if it improves their performance.

In [ ]:
# LINEAR REGRESSION SCALED

scaler = StandardScaler()
scaled_array = scaler.fit_transform(df_modeled)
df_scaled = pd.DataFrame(scaled_array, columns=df_modeled.columns)

linear_regression = regression_models.Regression(df_scaled, test_size=.2, random_state=3)
X = df_scaled.drop(columns=[])
y = df_scaled['']
y_pred, y_test, X_test, model, X = linear_regression.linear_regression(X, y)
linear_regression.print_results(y_pred, y_test, X_test, model)

Scaling the linear regression model had no effect on its performance, which is expected since linear regression is not sensitive to feature scaling. This approach will not be included within my ML model as its current performance is satisfactory without scaling.

---
## 8 · Model Evaluation

| Metric | Business Meaning |
|---|---|
| **RMSE** | Penalises large mispricings — direct financial loss per deal |
| **MAE** | Average dollar error on buy-price offers |
| **R²** | Proportion of price variance explained |
| **MAPE** | Must be < discount rate for the company to profit |

#### 8.1 · Model Comparison & Back-Testing

The train-test-split strategy was also explored in my ML pipeline, and the 80/20 split provided the best performance for the models.

Example of 50/50 train-test-split --> random_forest = regression_models.Regression(df_modeled, `test_size=.5`, random_state=3)

Example of 95/5 train-test-split --> random_forest = regression_models.Regression(df_modeled, `test_size=.05`, random_state=3)